# PRT-DeepONet — Irreversible sorption (concentration variant, full training pipeline)

Trains the steady-state concentration DeepONet for irreversible sorption. It is **velocity-
conditioned** (CVB): the velocity variant's predicted velocity `(ux, uy)` (λ=10) is z-scored and
fed into the geometry branch.

**Branch CNN** ← `[mask, ux, uy]` (predicted velocity, z-scored).  
**Branch FNN** ← the parametric conditions `(Pe, Da)` (min-max normalized).  
**Trunk** ← `(x, y)` and the **GDF** (Geodesic Distance Function, inlet distance).

Target: the steady-state concentration field `A ∈ [0, 1]`. The released checkpoint is
`../parameters/Irreversible_Sorption.pt`. Concentration is evaluated by **RMSE** over the porous ROI.


In [ ]:
# ====== 0. Imports ======
import os, copy, time, json
from collections import deque
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
# ====== 1. Reproducibility & Paths ======
import os
nx, ny = 64, 148
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED, BATCH, LR, EPOCHS, PATIENCE = 42, 25, 1e-3, 1000, 15

# All inputs live under one data root — set $PRT_DATA_ROOT or edit DATA_ROOT.
# Expected layout is documented in the README ("Data layout").
DATA_ROOT = os.environ.get('PRT_DATA_ROOT', '../../data')
NPZ       = f'{DATA_ROOT}/concentration/irreversible'   # A{dom}.npz (label) + m{b}.npz (mask)
VELC_PATH = f'{DATA_ROOT}/predicted_velocity.npz'       # predicted velocity (ux,uy), λ=10
SPLIT_PT  = f'{DATA_ROOT}/velocity/split.pt'            # fixed porosity-binned split (seed 27)
MODEL_OUT = '../parameters/Irreversible_Sorption.pt'       # trained output
STATS_OUT = '../parameters/Irreversible_Sorption_stats.json'
BAD_BASES = {1357, 1675, 2637}
DAmin, DAmax, PEmin, PEmax = 74, 740, 1, 10


In [ ]:
# ====== 2. Model Definition (paper-style names) ======
class CustomCNN(nn.Module):
    def __init__(s, in_channels, out_dim, num_blocks=5):
        super().__init__(); ch=[in_channels,16,32,64,128,256][:num_blocks+1]; L=[]
        for i in range(num_blocks): L+=[nn.Conv2d(ch[i],ch[i+1],3,1,1), nn.SiLU(), nn.AvgPool2d(2)]
        s.features=nn.Sequential(*L); h,w=64,148
        for _ in range(num_blocks): h//=2; w//=2
        s.fc=nn.Linear(ch[num_blocks]*h*w, out_dim)
    def forward(s, x): x=s.features(x); return s.fc(x.view(x.size(0),-1))
class PeDaMLP(nn.Module):
    """Parametric branch over the (Pe, Da) conditions."""
    def __init__(s, in_dim=2, out_dim=128, hidden=128, layers=3):
        super().__init__(); m=[nn.Linear(in_dim,hidden), nn.SiLU()]
        for _ in range(layers-2): m+=[nn.Linear(hidden,hidden), nn.SiLU()]
        m+=[nn.Linear(hidden,out_dim)]; s.net=nn.Sequential(*m)
    def forward(s, x): return s.net(x)
def make_trunk(in_dim, out_dim=128, layers=8, width=128):
    m=[nn.Linear(in_dim,width), nn.SiLU()]
    for _ in range(layers-2): m+=[nn.Linear(width,width), nn.SiLU()]
    m+=[nn.Linear(width,out_dim)]; return nn.Sequential(*m)
class GeneralDeepONet(nn.Module):
    """branch1=[mask,ux,uy] (CNN) | branch2=[Pe,Da] (MLP) | trunk=[x,y,GDF] -> concentration."""
    def __init__(s, branch1, b2_in=2, tr_in=3, out_dim=128):
        super().__init__(); s.branch1_net=branch1; s.branch2_net=PeDaMLP(b2_in,out_dim,128,3)
        s.trunk_net=make_trunk(tr_in,out_dim,8,128); s.bias=nn.Parameter(torch.zeros(1)); s.nx,s.ny=nx,ny
    def forward(s, b1, b2, tr):
        N,T,Lp,D=tr.shape; to=s.trunk_net(tr.view(-1,D)).view(N,Lp,-1).unsqueeze(1)
        b1o=s.branch1_net(b1).unsqueeze(1).unsqueeze(2); b2o=s.branch2_net(b2).unsqueeze(1).unsqueeze(2)
        return ((b1o*b2o*to).sum(-1)+s.bias).view(N,s.nx,s.ny,1)


In [ ]:
# ====== 3. Data Loader ======
# GDF: Geodesic Distance Function (inlet distance), inlet-high / solid=0
def boundary_distance_map(m):
    nxx,nyy=m.shape; dist=np.zeros_like(m,dtype=np.float32)
    starts=[(i,0) for i in range(nxx) if m[i,0]==1]
    for (x,y) in starts: dist[x,y]=0.5
    q=deque(starts)
    while q:
        x,y=q.popleft()
        for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            a,b=x+dx,y+dy
            if 0<=a<nxx and 0<=b<nyy and m[a,b]==1:
                nd=dist[x,y]+1.0
                if dist[a,b]==0 or dist[a,b]>nd: dist[a,b]=nd; q.append((a,b))
    return dist
def make_inlet_distance_norm(m):
    d=boundary_distance_map(m); mx=np.nanmax(d[m==1]); d[m==0]=mx+1; d=-d
    v=np.isfinite(d); lo,hi=d[v].min(),d[v].max(); out=np.zeros_like(d)
    if hi>lo: out[v]=(d[v]-lo)/(hi-lo)
    out[m==0]=0; return out.astype(np.float32)

def get_Da_Pe(idx):
    Da=74 if idx<=9000 else (296 if idx<=18000 else 740)
    r=((idx-1)%9000); Pe=1 if r<3000 else (5 if r<6000 else 10); return Da,Pe
def mmv(v,a,b): return (v-a)/(b-a)
def domains_for(bases): return [g*3000+b for b in bases for g in range(9)]

def load_dataset():
    # --- fixed split (reuse the velocity split; drop known-bad bases) ---
    V=torch.load(SPLIT_PT, map_location='cpu', weights_only=False)
    def vbase(i): return ((int(i)-1)%3000)+1
    test_bases =sorted({vbase(s['idx']) for s in V['test']}  - BAD_BASES)
    train_bases=sorted({vbase(s['idx']) for s in V['train']} - BAD_BASES); del V
    train_doms=domains_for(train_bases); test_doms=domains_for(test_bases)
    need_bases=sorted(set(train_bases)|set(test_bases))

    # --- geometry: mask + GDF (inlet distance) per base ---
    MASK={}; DINLET={}
    for b in need_bases:
        with np.load(os.path.join(NPZ,f'm{b}.npz')) as d: m=(d['m'].reshape(nx,ny)>0.5).astype(int)
        MASK[b]=m; DINLET[b]=make_inlet_distance_norm(m)
    xs=np.arange(nx,dtype=np.float32)/(nx-1); ys=np.arange(ny,dtype=np.float32)/(ny-1)
    Xg,Yg=np.meshgrid(xs,ys,indexing='ij'); Xf=Xg.flatten(); Yf=Yg.flatten()

    # --- predicted velocity (ux,uy), z-scored over train pore cells ---
    VELC=np.load(VELC_PATH); grp=VELC['GROUP_MEANRE']; _o=list(np.argsort(grp))
    Pe2vg={1:int(_o[0]),5:int(_o[1]),10:int(_o[2])}   # Pe -> velocity group (by mean Re)
    def velfield(base,Pe): return VELC[str(Pe2vg[Pe]*3000+base)]
    tu=np.concatenate([velfield(b,pe)[0][MASK[b]==1] for b in train_bases for pe in (1,5,10)])
    tv=np.concatenate([velfield(b,pe)[1][MASK[b]==1] for b in train_bases for pe in (1,5,10)])
    UMU,USD=float(tu.mean()),float(tu.std()); VMU,VSD=float(tv.mean()),float(tv.std()); del tu,tv
    def u_z(b,pe): return ((velfield(b,pe)[0]-UMU)/USD).astype(np.float32)
    def v_z(b,pe): return ((velfield(b,pe)[1]-VMU)/VSD).astype(np.float32)

    # --- concentration labels A{dom}.npz ---
    Acache={}
    def load_A(dom):
        if dom in Acache: return Acache[dom]
        with np.load(os.path.join(NPZ,f'A{dom}.npz')) as d:
            A=d['A']; A=(A.reshape(nx,ny) if A.ndim==1 else A[-1,:].reshape(nx,ny)).astype(np.float32)
        Acache[dom]=A; return A

    def build(doms):
        N=len(doms); Lp=nx*ny
        b1=np.empty((N,3,nx,ny),np.float32); b2=np.empty((N,2),np.float32)
        tr=np.empty((N,1,Lp,3),np.float32); yt=np.empty((N,nx,ny,1),np.float32); ms=[]
        for i,dom in enumerate(doms):
            b=((dom-1)%3000)+1; Da,Pe=get_Da_Pe(dom); m=MASK[b]; ms.append(m)
            b1[i,0]=m; b1[i,1]=u_z(b,Pe); b1[i,2]=v_z(b,Pe)             # [mask, ux, uy]
            b2[i,0]=mmv(Da,DAmin,DAmax); b2[i,1]=mmv(Pe,PEmin,PEmax)    # [Da_n, Pe_n]
            tr[i,0]=np.stack([Xf,Yf,DINLET[b].flatten()],axis=1)        # [x, y, GDF]
            yt[i,...,0]=load_A(dom)
        return (torch.from_numpy(b1),torch.from_numpy(b2),torch.from_numpy(tr),torch.from_numpy(yt),ms)
    tb1,tb2,ttr,tyt,_   = build(train_doms)
    eb1,eb2,etr,eyt,ems = build(test_doms)
    stats=dict(UMU=UMU,USD=USD,VMU=VMU,VSD=VSD,Pe2vg=Pe2vg)
    return (tb1,tb2,ttr,tyt),(eb1,eb2,etr,eyt,ems),stats


In [ ]:
# ====== 4. Training Utilities ======
def train_model(model, train_tensors, test_tensors, num_epochs=EPOCHS, lr=LR, batch_size=BATCH, patience=PATIENCE):
    tb1,tb2,ttr,tyt=train_tensors; eb1,eb2,etr,eyt=test_tensors[:4]
    tl=DataLoader(TensorDataset(tb1,tb2,ttr,tyt),batch_size=batch_size,shuffle=True)
    vl=DataLoader(TensorDataset(eb1,eb2,etr,eyt),batch_size=batch_size,shuffle=False)
    opt=torch.optim.AdamW(model.parameters(),lr=lr); crit=nn.HuberLoss(delta=1.0)
    scaler=torch.cuda.amp.GradScaler()
    best=None; bestloss=1e9; bestep=0; noimp=0; t0=time.time()
    for ep in range(1,num_epochs+1):
        model.train()
        for bb in tl:
            b1,b2,tr,y=[x.to(device) for x in bb]; opt.zero_grad()
            with torch.cuda.amp.autocast(): loss=crit(model(b1,b2,tr),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        model.eval(); tot=0.0; n=0
        with torch.no_grad():
            for bb in vl:
                b1,b2,tr,y=[x.to(device) for x in bb]
                with torch.cuda.amp.autocast(): tot+=float(crit(model(b1,b2,tr),y).item()); n+=1
        vloss=tot/n
        if vloss<bestloss-1e-5: bestloss=vloss; bestep=ep; noimp=0; best=copy.deepcopy(model.state_dict())
        else:
            noimp+=1
            if noimp>=patience: break
        if ep%10==0 or ep<=2: print(f'  ep{ep:3d} val{vloss:.6f} best{bestloss:.6f}@{bestep} {time.time()-t0:.0f}s', flush=True)
    model.load_state_dict(best); return best, bestep


In [ ]:
# ====== 5. Evaluation Example (concentration RMSE over the porous ROI) ======
@torch.no_grad()
def evaluate(model, test_tensors, num_samples=5):
    eb1,eb2,etr,eyt,ems=test_tensors; model.eval()
    for i in range(min(num_samples,eb1.shape[0])):
        pr=model(eb1[i:i+1].to(device),eb2[i:i+1].to(device),etr[i:i+1].to(device)).cpu().numpy().squeeze()
        gt=eyt[i].numpy().squeeze(); m=ems[i]
        rmse=float(np.sqrt(np.mean((pr[m==1]-gt[m==1])**2)))
        print(f'  sample {i}: concentration RMSE = {rmse:.4f}')


In [ ]:
# ====== 6. Main Entry ======
if __name__ == '__main__':
    # 1) Load data + build features (GDF, predicted velocity)
    train_tensors, test_tensors, stats = load_dataset()
    print('train/test:', train_tensors[0].shape[0], test_tensors[0].shape[0])
    # 2) Build model:  branch1=[mask,ux,uy] | branch2=[Pe,Da] | trunk=[x,y,GDF]
    np.random.seed(SEED); torch.manual_seed(SEED)
    if device.type=='cuda': torch.cuda.manual_seed_all(SEED)
    model = GeneralDeepONet(CustomCNN(3,128,5), b2_in=2, tr_in=3, out_dim=128).to(device)
    # 3) Train (dense Huber loss)
    best, bestep = train_model(model, train_tensors, test_tensors)
    # 4) Evaluate
    evaluate(model, test_tensors)
    # 5) Save parameters + velocity z-score stats -> ../parameters/Irreversible_Sorption.pt
    torch.save(best, MODEL_OUT)
    json.dump(stats, open(STATS_OUT,'w'), indent=1)
    print('[save]', MODEL_OUT, '| best epoch', bestep)
